**Principal Component Analysis**

You will implement dimensionality reduction with PCA.  

1). Read iris_dataset.csv (4 features, hence 4 PCs)

2). Find the principal components

3). Recontruct the dataset (X_hat)

4). Determine the accuracy of X_hat for 1 PC and 4 PCs using LDA classifier (provided below)


In [9]:
import numpy as np
import pandas as pd
#import matplotlib.pyplot as plt
from numpy import linalg as LA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_val_score


# Load data - 150 observations, 4 features, 3 classes,
df = pd.read_csv("iris_dataset.csv", header=None)
print(df.describe())
data = df.values
print(np.shape(data))

                0           1           2           3           4
count  150.000000  150.000000  150.000000  150.000000  150.000000
mean     5.843333    3.057333    3.758000    1.199333    1.000000
std      0.828066    0.435866    1.765298    0.762238    0.819232
min      4.300000    2.000000    1.000000    0.100000    0.000000
25%      5.100000    2.800000    1.600000    0.300000    0.000000
50%      5.800000    3.000000    4.350000    1.300000    1.000000
75%      6.400000    3.300000    5.100000    1.800000    2.000000
max      7.900000    4.400000    6.900000    2.500000    2.000000
(150, 5)


In [10]:
## Setup

# Shuffle data randomly
shuffled_data = data;
np.random.shuffle(shuffled_data)
X = shuffled_data[:,0:4]  # 150x4
y = shuffled_data[:,4]

# Classification accuracy with the original dataset using LDA
model_mean_scores = []
model = LinearDiscriminantAnalysis().fit(X, y)
scores = cross_val_score(model, X, y, cv=10)
model_mean_scores.append(np.mean(scores))
print('>> Average accuracy with the original dataset = {0:0.4f}'.format(model_mean_scores[0]))


>> Average accuracy with the original dataset = 0.9800


In [11]:
def evaluate_accuracy(X_hat, Num_PC, y):

  ###############################################
  # Evaluate classificatin accuracy with LDA
  ###############################################
  '''
    Inputs:
      X_hat: reconstructed dataset. dimension=150x4
      Num_PC: number of PC's used to recover X_hat
      y: class label vector. dimension=150x1

  '''

  from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
  from sklearn.model_selection import cross_val_score

  X_train = X_hat[:,0:Num_PC]        # dimensionally reduced dataset
  y_train = y

  model_mean_scores = []
  model = LinearDiscriminantAnalysis().fit(X_train, y_train)
  scores = cross_val_score(model, X_train, y_train, cv=10)
  model_mean_scores.append(np.mean(scores))

  print('Average accuracy = {0:0.4f} with {1:1d} PCs'
     .format(model_mean_scores[0], Num_PC))

In [13]:
### Your code goes here ...

## PCA Implementation using Eigen Decomposition

# Step 1: Center the data (subtract mean)
X_mean = np.mean(X, axis=0)
X_centered = X - X_mean

# Step 2: Compute the covariance matrix
C = np.cov(X_centered.T)  # 4x4 covariance matrix

# Step 3: Compute eigenvalues and eigenvectors using eigen decomposition
eigenvalues, eigenvectors = LA.eigh(C)

# Step 4: Sort eigenvectors by eigenvalues in descending order
sorted_idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[sorted_idx]
eigenvectors = eigenvectors[:, sorted_idx]  # columns are eigenvectors (PCs)

print("Eigenvalues:", eigenvalues)
print("Eigenvectors shape:", eigenvectors.shape)

# Step 5: Project data onto principal components (all 4 PCs)
# Z is the PCA-transformed data (scores in PCA space)
Z = X_centered @ eigenvectors  # 150x4 - each column is a PC score

# Step 6: Reconstruct the data set X_hat for 1 PC and 4 PCs
# X_hat = Z[:, :k] @ eigenvectors[:, :k].T + X_mean

# For 1 PC
k = 1
X_hat_1pc = Z[:, :k] @ eigenvectors[:, :k].T + X_mean
reconstruction_error_1pc = np.mean(np.sum((X - X_hat_1pc)**2, axis=1))
print(f"\nReconstruction error with 1 PC: {reconstruction_error_1pc:.4f}")

# For 4 PCs
k = 4
X_hat_4pc = Z[:, :k] @ eigenvectors[:, :k].T + X_mean
reconstruction_error_4pc = np.mean(np.sum((X - X_hat_4pc)**2, axis=1))
print(f"Reconstruction error with 4 PCs: {reconstruction_error_4pc:.6f}")

# X_hat = Z (the projected data in PCA space)
# evaluate_accuracy uses X_hat[:,0:Num_PC], so X_hat must be the PCA projections
X_hat = Z  # 150x4 matrix where each column is a principal component score

## Use function evaluate_accuracy
print("\n--- LDA Classification Accuracy on PCA-projected data ---")
evaluate_accuracy(X_hat, 1, y)  # classification accuracy with 1 PC
evaluate_accuracy(X_hat, 4, y)  # classification accuracy with 4 PCs

Eigenvalues: [4.22824171 0.24267075 0.0782095  0.02383509]
Eigenvectors shape: (4, 4)

Reconstruction error with 1 PC: 0.3424
Reconstruction error with 4 PCs: 0.000000

--- LDA Classification Accuracy on PCA-projected data ---
Average accuracy = 0.9267 with 1 PCs
Average accuracy = 0.9800 with 4 PCs
